# 02 - Fetch and Merge ACS Variables for CRD Mortality

This notebook fetches selected variables from the ACS Census API for years 2012-2019 and merges them with preprocessed CRD mortality data.

## Variables Selected (19 total):

### Standard Tables - B (14 variables):
- Median Household Income
- Total Population
- Gini Index
- Median Age
- Hispanic Population
- Black Population
- White Population
- No Vehicle (Owner)
- No Vehicle (Renter)
- Total Occupied Households
- Rent Burden Count (+50%)
- Rent Denominator
- Total Families (Single Mother)
- Total Families

### Summary Tables - S (5 variables):
- Poverty Rate
- Unemployment Rate
- Disability Rate
- Bachelor's Degree or Higher (%)
- High School Degree or Higher (%)

**Note:** Variable names are in Title Case format (publication-ready). Raw counts will be converted to percentages in later notebooks.

## 1. Import Libraries

In [ ]:
import pandas as pd
import requests
import os

## 2. API Key Configuration

Set your Census API key as an environment variable for security:
```bash
export CENSUS_API_KEY='your_api_key_here'
```

In [ ]:
API_KEY = os.getenv('CENSUS_API_KEY')

if API_KEY is None:
    API_KEY = 'YOUR_API_KEY_HERE'
    print("Warning: Using hardcoded API key. Consider using environment variable for security.")
else:
    print("API key loaded from environment variable")

## 3. Function Definitions

Two functions to handle different ACS table types: B (standard) and S (summary).

### 3.1 Function for Standard Tables (B)

In [ ]:
def fetch_and_merge_acs_variable(variable_code, variable_name, year, api_key):
    preprocessed_path = f'../data/processed/preprocessed_fips_crd/preprocessed_crd_fips_{year}.csv'
    output_path = f'../data/processed/acs_individual_variables/dataset_with_{variable_name}_{year}.csv'

    print(f"Loading preprocessed data for {year}...")
    preprocessed_df = pd.read_csv(preprocessed_path)

    print(f"Fetching {variable_name} ({variable_code}) from Census API for {year}...")
    acs_endpoint = f'https://api.census.gov/data/{year}/acs/acs5'
    params = {
        'get': variable_code,
        'for': 'county:*',
        'in': 'state:*',
        'key': api_key
    }

    response = requests.get(acs_endpoint, params=params)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch {variable_name}. Status code: {response.status_code}")

    acs_data = response.json()
    acs_df = pd.DataFrame(columns=acs_data[0], data=acs_data[1:])
    acs_df = acs_df.rename(columns={
        variable_code: variable_name,
        'state': 'State_FIPS',
        'county': 'County_FIPS'
    })

    preprocessed_df['State_FIPS'] = preprocessed_df['State_FIPS'].astype(str).str.zfill(2)
    preprocessed_df['County_FIPS'] = preprocessed_df['County_FIPS'].astype(str).str.zfill(3)
    acs_df['State_FIPS'] = acs_df['State_FIPS'].str.zfill(2)
    acs_df['County_FIPS'] = acs_df['County_FIPS'].str.zfill(3)

    acs_df[variable_name] = pd.to_numeric(acs_df[variable_name], errors='coerce')

    final_df = pd.merge(
        preprocessed_df,
        acs_df[['State_FIPS', 'County_FIPS', variable_name]],
        on=['State_FIPS', 'County_FIPS'],
        how='left'
    )

    if 'crd_mortality_rate' not in final_df.columns:
        raise ValueError("'crd_mortality_rate' column is missing in the final merged dataset.")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    final_df.to_csv(output_path, index=False)
    print(f"  Saved: {output_path}")

    return final_df

### 3.2 Function for Summary Tables (S)

In [ ]:
def fetch_and_merge_acs_variable_summary(variable_code, variable_name, year, api_key):
    preprocessed_path = f'../data/processed/preprocessed_fips_crd/preprocessed_crd_fips_{year}.csv'
    output_path = f'../data/processed/acs_individual_variables/dataset_with_{variable_name}_{year}.csv'

    print(f"Loading preprocessed data for {year}...")
    preprocessed_df = pd.read_csv(preprocessed_path)

    print(f"Fetching {variable_name} ({variable_code}) from Census API for {year}...")
    acs_endpoint = f'https://api.census.gov/data/{year}/acs/acs5/subject'
    params = {
        'get': variable_code,
        'for': 'county:*',
        'in': 'state:*',
        'key': api_key
    }

    response = requests.get(acs_endpoint, params=params)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch {variable_name}. Status code: {response.status_code}")

    acs_data = response.json()
    acs_df = pd.DataFrame(columns=acs_data[0], data=acs_data[1:])
    acs_df = acs_df.rename(columns={
        variable_code: variable_name,
        'state': 'State_FIPS',
        'county': 'County_FIPS'
    })

    preprocessed_df['State_FIPS'] = preprocessed_df['State_FIPS'].astype(str).str.zfill(2)
    preprocessed_df['County_FIPS'] = preprocessed_df['County_FIPS'].astype(str).str.zfill(3)
    acs_df['State_FIPS'] = acs_df['State_FIPS'].str.zfill(2)
    acs_df['County_FIPS'] = acs_df['County_FIPS'].str.zfill(3)

    acs_df[variable_name] = pd.to_numeric(acs_df[variable_name], errors='coerce')

    final_df = pd.merge(
        preprocessed_df,
        acs_df[['State_FIPS', 'County_FIPS', variable_name]],
        on=['State_FIPS', 'County_FIPS'],
        how='left'
    )

    if 'crd_mortality_rate' not in final_df.columns:
        raise ValueError("'crd_mortality_rate' column is missing in the final merged dataset.")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    final_df.to_csv(output_path, index=False)
    print(f"  Saved: {output_path}")

    return final_df

## 4. Fetch Standard Variables (B Tables)

Demographic and economic variables for all counties, 2012-2019.

In [ ]:
standard_variables = [
    {'code': 'B19013_001E', 'name': 'Median Household Income'},
    {'code': 'B01003_001E', 'name': 'Total Population'},
    {'code': 'B19083_001E', 'name': 'Gini Index'},
    {'code': 'B01002_001E', 'name': 'Median Age'},
    {'code': 'B03003_003E', 'name': 'Hispanic Population'},
    {'code': 'B02001_003E', 'name': 'Black Population'},
    {'code': 'B02001_002E', 'name': 'White Population'},
    {'code': 'B25044_003E', 'name': 'No Vehicle (Owner)'},
    {'code': 'B25044_010E', 'name': 'No Vehicle (Renter)'},
    {'code': 'B25044_001E', 'name': 'Total Occupied Households'},
    {'code': 'B25070_010E', 'name': 'Rent Burden Count (+50%)'},
    {'code': 'B25070_001E', 'name': 'Rent Denominator'},
    {'code': 'B11003_016E', 'name': 'Total Families (Single Mother)'},
    {'code': 'B11003_001E', 'name': 'Total Families'}
]

for year in range(2012, 2020):
    print(f"\n--- Year {year} ---")
    for var in standard_variables:
        fetch_and_merge_acs_variable(
            variable_code=var['code'],
            variable_name=var['name'],
            year=year,
            api_key=API_KEY
        )

print("\nAll standard variables fetched successfully!")

## 5. Fetch Summary Table Variables (S Tables)

Health, social, and education variables.

In [ ]:
summary_variables = [
    {'code': 'S1701_C03_001E', 'name': 'Poverty Rate'},
    {'code': 'S2301_C04_001E', 'name': 'Unemployment Rate'},
    {'code': 'S1810_C03_001E', 'name': 'Disability Rate'},
    {'code': 'S1501_C02_015E', 'name': "Bachelor's Degree or Higher (%)"},
    {'code': 'S1501_C02_014E', 'name': 'High School Degree or Higher (%)'}
]

print("=" * 60)
print("FETCHING SUMMARY TABLE VARIABLES (S Tables)")
print("=" * 60)

for year in range(2012, 2020):
    print(f"\n--- Year {year} ---")
    for var in summary_variables:
        fetch_and_merge_acs_variable_summary(
            variable_code=var['code'],
            variable_name=var['name'],
            year=year,
            api_key=API_KEY
        )

print("\nAll summary table variables fetched successfully!")

## 6. Summary

All ACS variables have been fetched and merged with CRD mortality data for years 2012-2019.

**Output location:** `../data/processed/acs_individual_variables/`

**Total files created:** 19 variables × 8 years = 152 CSV files

Each file contains:
- County and State information
- State_FIPS and County_FIPS codes
- Fips (full 5-digit FIPS code)
- `crd_mortality_rate` (target variable)
- The corresponding ACS variable (in Title Case)

**Next step:** Proceed to notebook 03 to combine all features into a single dataset for each year.

In [ ]:
output_dir = '../data/processed/acs_individual_variables/'
if os.path.exists(output_dir):
    files = [f for f in os.listdir(output_dir) if f.endswith('.csv')]
    print(f"Total files created: {len(files)}")
    print(f"\nSample files:")
    for f in sorted(files)[:5]:
        print(f"  {f}")